In [1]:
!pip install mlflow scikit-learn pandas numpy

In [2]:
import os
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("[INFO] Librerías importadas correctamente.")
print(f"[INFO] Versión de MLflow: {mlflow.__version__}")

[INFO] Librerías importadas correctamente.
[INFO] Versión de MLflow: 3.17.1


In [3]:
# Crear directorios si no existen
os.makedirs("c:/Users/Luis/Downloads/Actividad5/datos/datos_ini", exist_ok=True)
os.makedirs("c:/Users/Luis/Downloads/Actividad5/datos/datos_limp", exist_ok=True)

# Cargar dataset público (Breast Cancer Wisconsin)
data = load_breast_cancer(as_frame=True)
df = data.frame

# Guardar datos originales
path_ini = "../datos/datos_ini/cancer_original.csv"
df.to_csv(path_ini, index=False)
print(f"Datos originales guardados en: {path_ini}")

# Preprocesamiento y Limpieza básica
df = df.dropna()
X = df.drop(columns=["target"])
y = df["target"]

# División train/test (80/20) estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalado de características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Guardar datos limpios procesados
df_train_limp = pd.DataFrame(X_train_scaled, columns=X.columns)
df_train_limp["target"] = y_train.values

df_test_limp = pd.DataFrame(X_test_scaled, columns=X.columns)
df_test_limp["target"] = y_test.values

train_path = "../datos/datos_limp/train_limpio.csv"
test_path = "../datos/datos_limp/test_limpio.csv"

df_train_limp.to_csv(train_path, index=False)
df_test_limp.to_csv(test_path, index=False)

print(f"Datos limpios guardados en: {train_path} y {test_path}")

Datos originales guardados en: ../datos/datos_ini/cancer_original.csv
Datos limpios guardados en: ../datos/datos_limp/train_limpio.csv y ../datos/datos_limp/test_limpio.csv


In [5]:
# Configurar experimento en MLflow local
mlflow.set_experiment("Actividad5_Clasificacion_Colab")

# Definir modelos y espacios de búsqueda para GridSearchCV
modelos = {
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=1000, random_state=42),
        "params": {
            "C": [0.01, 0.1, 1.0, 10.0],
            "solver": ["lbfgs"]
        }
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [50, 100, 200],
            "max_depth": [None, 5, 10],
            "min_samples_split": [2, 5]
        }
    }
}

resultados = []

for nombre, config in modelos.items0 if False else modelos.items():
    print(f"\n----------------------------------------")
    print(f"Entrenando y optimizando: {nombre}")
    print(f"----------------------------------------")
    
    with mlflow.start_run(run_name=nombre):
        # Validación cruzada con GridSearchCV (5-Fold)
        grid_search = GridSearchCV(
            estimator=config["model"],
            param_grid=config["params"],
            cv=5,
            scoring="f1",
            n_jobs=-1
        )
        
        grid_search.fit(X_train_scaled, y_train)
        
        # Mejor modelo y sus hiperparámetros
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_
        
        # Evaluación en el conjunto de prueba (Test)
        y_pred = best_model.predict(X_test_scaled)
        y_proba = best_model.predict_proba(X_test_scaled)[:, 1] if hasattr(best_model, "predict_proba") else y_pred

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)

        # Registro de hiperparámetros y métricas en MLflow
        mlflow.log_params(best_params)
        mlflow.log_param("algoritmo", nombre)
        
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", roc_auc)

        # Registro del artefacto del modelo
        mlflow.sklearn.log_model(
    best_model, 
    artifact_path=f"modelo_{nombre}", 
    serialization_format="cloudpickle"
)
        
        # Guardar en memoria para mostrar tabla resumen
        resultados.append({
            "Algoritmo": nombre,
            "Mejores Parámetros": str(best_params),
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1-Score": f1,
            "ROC-AUC": roc_auc
        })
        
        print(f"[OK] {nombre} registrado con éxito en MLflow.")
        print(f"  -> F1-Score: {f1:.4f} | Accuracy: {acc:.4f} | ROC-AUC: {roc_auc:.4f}")

# Mostrar tabla comparativa final
df_resultados = pd.DataFrame(resultados)
display(df_resultados)

2026/10/16 23:12:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/16 23:12:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



----------------------------------------
Entrenando y optimizando: LogisticRegression
----------------------------------------
[OK] LogisticRegression registrado con éxito en MLflow.
  -> F1-Score: 0.9793 | Accuracy: 0.9737 | ROC-AUC: 0.9957

----------------------------------------
Entrenando y optimizando: RandomForest
----------------------------------------


2026/10/16 23:12:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/16 23:12:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[OK] RandomForest registrado con éxito en MLflow.
  -> F1-Score: 0.9655 | Accuracy: 0.9561 | ROC-AUC: 0.9932


,Algoritmo,Mejores Parámetros,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,LogisticRegression,"{'C': 0.1, 'solver': 'lbfgs'}",0.973684,0.972603,0.986111,0.979310,0.995701
1,RandomForest,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.956140,0.958904,0.972222,0.965517,0.993221
